In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import cv2
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
df = pd.read_csv('/content/drive/MyDrive/monkeytype/monkeytyping.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6159 entries, 0 to 6158
Data columns (total 31 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   participant_id             6159 non-null   object 
 1   date                       6159 non-null   object 
 2   original_filename          6159 non-null   object 
 3   trial_number               6159 non-null   int64  
 4   reaction_time              6159 non-null   object 
 5   correct                    6159 non-null   int64  
 6   correct_position           6159 non-null   object 
 7   chosen_position            6159 non-null   object 
 8   prompt                     0 non-null      float64
 9   sample_name                0 non-null      float64
 10  file_1_name                6159 non-null   object 
 11  file_2_name                6159 non-null   object 
 12  file_3_name                6139 non-null   object 
 13  file_4_name                6139 non-null   objec

In [3]:
df_symbols = df[df['colored'].isna()].copy(deep=True)
df_symbols = df_symbols[df_symbols['file_3_name'].notna()].copy(deep=True)
df_symbols = df_symbols[df_symbols['chosen_position']!='No decision'].copy(deep=True)
#df_symbols.info()
image_filepath_main = '/content/drive/MyDrive/monkeytype/emnist_rare_ESRGAN_improved/'
file_cols_name = ['file_1_name', 'file_2_name', 'file_3_name', 'file_4_name']
for f in file_cols_name:
  df_symbols[f] = df_symbols[f].astype(str)
  for index, row in df_symbols.iterrows():
    df_symbols.loc[index, f] = df_symbols.loc[index, f][0] + '/'+df_symbols.loc[index, f][2:] + '_out.png'
print(df_symbols[file_cols_name[0]])

0       y/00902_out.png
1       s/00320_out.png
2       c/01270_out.png
3       f/00745_out.png
4       s/01180_out.png
             ...       
6154    f/01252_out.png
6155    m/01187_out.png
6156    m/00439_out.png
6157    m/00413_out.png
6158    c/00117_out.png
Name: file_1_name, Length: 4675, dtype: object


In [4]:
df_files = df_symbols[file_cols_name].copy(deep = True)
df_files.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4675 entries, 0 to 6158
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   file_1_name  4675 non-null   object
 1   file_2_name  4675 non-null   object
 2   file_3_name  4675 non-null   object
 3   file_4_name  4675 non-null   object
dtypes: object(4)
memory usage: 311.7+ KB


In [5]:
symbols_list = ['m', 'c', 's', 'y', 'f', 'j']
new_file_cols = ['file_1_symb', 'file_2_symb', 'file_3_symb', 'file_4_symb']
for index, row in df_files.iterrows():
  for f, fs in zip(file_cols_name, new_file_cols):
    df_files.loc[index, fs] = df_files.loc[index, f][0]
df_files.head()

,file_1_name,file_2_name,file_3_name,file_4_name,file_1_symb,file_2_symb,file_3_symb,file_4_symb
0,y/00902_out.png,c/00891_out.png,f/01542_out.png,m/00280_out.png,y,c,f,m
1,s/00320_out.png,f/01788_out.png,m/01766_out.png,j/01080_out.png,s,f,m,j
2,c/01270_out.png,y/00523_out.png,j/01415_out.png,m/00902_out.png,c,y,j,m
3,f/00745_out.png,s/00125_out.png,m/01681_out.png,c/00697_out.png,f,s,m,c
4,s/01180_out.png,y/00443_out.png,j/01331_out.png,m/00367_out.png,s,y,j,m


In [6]:
for index, row in df_files.iterrows():
  for f in file_cols_name:
    df_files.loc[index, f] = image_filepath_main + df_files.loc[index, f]
df_files.head()

,file_1_name,file_2_name,file_3_name,file_4_name,file_1_symb,file_2_symb,file_3_symb,file_4_symb
0,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,y,c,f,m
1,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,s,f,m,j
2,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,c,y,j,m
3,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,f,s,m,c
4,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,s,y,j,m


In [7]:
brightness_columns = ['brightness0', 'brightness1', 'brightness2', 'brightness3']
local_symbol_columns = ['local_symbol_cls0', 'local_symbol_cls1', 'local_symbol_cls2', 'local_symbol_cls3',]
dataset_columns = ['participant_id',
                   'brightness0', 'brightness1', 'brightness2', 'brightness3',
                   'local_symbol_cls0', 'local_symbol_cls1', 'local_symbol_cls2', 'local_symbol_cls3',
                   'correct_symbol_cls',
                   'correct_local_cls',
                   'participant_linspace',
                   'time_reaction_task',
                   'time_reaction_mean',
                   'chosen_local_cls']
df_dataset = pd.DataFrame(columns = dataset_columns)
df_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   participant_id        0 non-null      object
 1   brightness0           0 non-null      object
 2   brightness1           0 non-null      object
 3   brightness2           0 non-null      object
 4   brightness3           0 non-null      object
 5   local_symbol_cls0     0 non-null      object
 6   local_symbol_cls1     0 non-null      object
 7   local_symbol_cls2     0 non-null      object
 8   local_symbol_cls3     0 non-null      object
 9   correct_symbol_cls    0 non-null      object
 10  correct_local_cls     0 non-null      object
 11  participant_linspace  0 non-null      object
 12  time_reaction_task    0 non-null      object
 13  time_reaction_mean    0 non-null      object
 14  chosen_local_cls      0 non-null      object
dtypes: object(15)
memory usage: 132.0+ bytes


In [8]:
for index, row in df_symbols.iterrows():
  df_dataset.loc[index, 'participant_id'] = df_symbols.loc[index, 'participant_id']
  df_dataset.loc[index, 'chosen_local_cls'] = int(df_symbols.loc[index, 'chosen_position'][0])-1
  df_dataset.loc[index, 'correct_local_cls'] = int(df_symbols.loc[index, 'correct_position'][0])-1
  df_dataset.loc[index, 'time_reaction_task'] = df_symbols.loc[index, 'reaction_time']
  df_dataset.loc[index, 'time_reaction_mean'] = df_symbols.loc[index, 'mean_reaction_time']
  for f, fs, bc in zip(file_cols_name, new_file_cols, brightness_columns):
    img = cv2.imread(df_files.loc[index, f], cv2.IMREAD_GRAYSCALE)
    imgf = img.astype(np.float32) / 255.0
    if img is None:
      print(f"Ошибка: не удалось загрузить изображение по пути: {df_files.loc[index, f]}")
    df_dataset.loc[index, bc] = np.mean(imgf)
  if index%100==0:
    print(index)

0
100
400
500
600
700
1000
1100
1200
1300
1400
1500
1700
2000
2200
2300
2400
2500
2600
2700
2800
2900
3100
3200
3300
3400
3600
3700
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100


In [9]:
for index, row in df_symbols.iterrows():
  symbol_cls_correct = df_symbols.loc[index, 'correct_position'][0]
  for fs, lscs in zip(new_file_cols, local_symbol_columns):
    df_dataset.loc[index, lscs] = str(df_files.loc[index, fs])
    if symbol_cls_correct in fs:
      df_dataset.loc[index, 'correct_symbol_cls'] = str(df_files.loc[index, fs])

In [10]:
g = df_symbols.groupby('participant_id')
pos = g.cumcount()

# размер группы (кол-во строк в сессии), "растянутый" до длины df
cnt = g['reaction_time'].transform('size')  # any column, просто чтобы вызвать transform

# чтобы получить [0, 1] включительно: pos / (cnt-1)
# аккуратно обходим деление на 0, если в группе всего одна строка
denom = (cnt - 1).where(cnt > 1, 1)  # если cnt==1, делим на 1 (получится 0)

df_dataset['participant_linspace'] = pos / denom

df_dataset

,participant_id,brightness0,brightness1,brightness2,brightness3,local_symbol_cls0,local_symbol_cls1,local_symbol_cls2,local_symbol_cls3,correct_symbol_cls,correct_local_cls,participant_linspace,time_reaction_task,time_reaction_mean,chosen_local_cls
0,Jupiter,0.181638,0.138126,0.138356,0.205347,y,c,f,m,c,1,0.000000,1029,980.0,2
1,Jupiter,0.220664,0.118452,0.17691,0.154541,s,f,m,j,f,1,0.000593,579,980.0,1
2,Jupiter,0.217307,0.12525,0.140104,0.199788,c,y,j,m,j,2,0.001186,688,980.0,3
3,Jupiter,0.132006,0.147323,0.11311,0.188276,f,s,m,c,c,3,0.001779,578,980.0,1
4,Jupiter,0.251793,0.16928,0.102143,0.135685,s,y,j,m,m,3,0.002372,848,980.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6154,Yunt,0.179335,0.168793,0.060612,0.254528,f,y,j,c,y,1,0.997557,1177,1424.0,1
6155,Yunt,0.152927,0.116505,0.119489,0.233521,m,c,y,s,m,0,0.998167,1132,1424.0,3
6156,Yunt,0.160991,0.23471,0.145027,0.171986,m,s,j,c,s,1,0.998778,1369,1424.0,2
6157,Yunt,0.214628,0.122164,0.117429,0.15146,m,y,j,c,y,1,0.999389,1623,1424.0,3


In [11]:
#df_dataset.to_csv('/content/drive/MyDrive/monkeytype/mt_dataset_catm.csv', index=False)

df_dataset.to_csv('/content/drive/MyDrive/monkeytype/mt_dataset_catm_b01.csv', index=False)